# Matched ablation study

Original D1 train/test animal assignments are preserved. MouseFull adds balanced D3 holdouts; rats keep all longitudinal scans of each animal together.

`USE_VALIDATION_FOR_SELECTION=False` trains the complete epoch budget without validation passes, uses a preset cosine learning-rate schedule, and evaluates the last epoch. Test animals are evaluated only after fitting.

Set it to `True` to reserve separate inner validation from the training pool and use ReduceLROnPlateau plus validation-selected weights. It never uses test animals for checkpoint selection. Start a fresh run when changing this policy.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments.training import configure_runtime

TRAINING_SEED: int = 42
runtime = configure_runtime(seed=TRAINING_SEED, memory_growth=True, mixed_precision=False)
runtime
import nibabel as nib

REFERENCE: Path = (
    REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49"
    / "Anat" / "TBI_fm_19_49_N4.nii.gz"
)
ref_img = nib.load(REFERENCE)
reference_axes = nib.aff2axcodes(ref_img.affine)
print("Reference axes:", reference_axes, "(left-right is the R/L axis)")


## Complete reference configuration

`MODE="fixed_split"` uses one common train/validation/test split. `MODE="cross_validation"` uses common grouped outer folds with separate checkpoint-selection validation. Split randomness stays fixed when `TRAINING_SEEDS` changes. Multiple seeds increase the number of fitted models, not the number of biological test animals.

The default matrix has four runs for a fixed split and one seed, or twenty fitted models for five-fold CV. Early stopping may produce different actual epoch counts despite the same maximum budget. Seeds are matched, but architecture changes can alter random initialization and augmentation streams; bitwise replay is not promised.

For MouseFull, explicitly substitute notebook 05's D1+D3 registration, eleven-class vocabulary, classifier and corrected loss. Do not mix datasets or losses within one architectural ablation matrix. Keep new variants scientifically distinct from dataset and initialization comparisons. This notebook performs no interpolation sensitivity analysis.

Full-volume outputs use `original_label_ids=True`. `postprocessing_settings` controls brain-mask cleanup and ventricle separation. Thresholds are voxel counts on the 0.1 mm grid; categorical ROI masks are not cleaned as binary masks. Merged D1/rat ventricles are separated before mapping to original IDs 3 (contra), 13 (ipsi), and 21 (third ventricle). MouseFull keeps its directly predicted sides. GT uses the selected original annotations, never the prediction postprocessor.

The historical `default_hemisphere="right"` selects the plane side containing array origin, rather than deriving anatomical right from the affine. It is used when no sufficiently large lesion is predicted. Splitting requires at least three predicted third-ventricle voxels; missing midline predictions raise an explicit scan-specific error.


In [ ]:
from typing import Literal

from src.experiments.config import (
    DataConfig, DatasetSpec, EvaluationPatchConfig, ExperimentConfig, FixedSplitConfig, InferenceConfig, InitializationConfig,
    LabelConfig, LossConfig, ModelConfig, PostprocessingConfig, PreprocessingConfig,
)
from src.evaluation.k_fold import TrainingConfig

DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
REFERENCE_SPLITS: Path | None = DATA_ROOT.parent / "D1_splits_final.json"
REFERENCE_FOLD: int = 4  # Fixed studies reuse this fold; CV uses all reference folds.
K_FOLD_COMPATIBLE: bool = False  # True runs every fold for each study member.
MODE: Literal["fixed_split", "cross_validation"] = "fixed_split"
if K_FOLD_COMPATIBLE:
    MODE = "cross_validation"
TRAINING_SEEDS: tuple[int, ...] = (42,)
RUN_STUDY: bool = True
RESUME_STUDY: bool = False
STUDY_DIR: Path = REPO_ROOT / "results" / "ablations" / f"d1_3drnet_components_reference_fold_{REFERENCE_FOLD}"
SPLIT_FILE: Path | None = None
FOLD_FILE: Path | None = None

# False: no validation pass, preset LR schedule, final epoch for evaluation.
USE_VALIDATION_FOR_SELECTION: bool = False
INITIAL_LEARNING_RATE: float = 0.001
MINIMUM_LEARNING_RATE: float = 1e-7

fixed_epoch_training: bool = not USE_VALIDATION_FOR_SELECTION
checkpoint_monitor: str = "loss"
if USE_VALIDATION_FOR_SELECTION:
    checkpoint_monitor = "val_loss"

training = TrainingConfig(
    model_name="d1_3drnet_ablation", k=5, epochs=100, batch_size=8,
    seed=TRAINING_SEED, data_multiplier=3, output_dir=str(REPO_ROOT / "results"),
    monitor=checkpoint_monitor, monitor_mode="min", fixed_epochs=fixed_epoch_training, default_callbacks=False,
    fit_kwargs={"verbose": 1},
)
# Used only when no reference or saved split is selected.
split_settings = FixedSplitConfig(
    validation_fraction=0.2, test_fraction=0.1,
)

data_settings = DataConfig(root=DATA_ROOT, datasets=(DatasetSpec(name="D1", annotation_scope="full_volume"),))
preprocessing = PreprocessingConfig(
    reference=REFERENCE, patch_size=(80, 80, 80), spacing_mm=(0.1, 0.1, 0.1),
    correct_units=True, augment=True, random_crop=True, crop_mode="center",
    # Native-grid MRI degradation; bounds retain their earlier magnitudes.
    intensity_augmentation_grid="native",
    blur_sigma_range=(0.0, 0.6), blur_sigma_units="voxels",
    noise_std_range=(0.0, 0.05), noise_std_units="intensity",
    flip_probability=0.5,
)
labels = LabelConfig(unknown="background")
# Sample once; validation/test reuse evaluation_patches.json across epochs and runs.
evaluation_patch_settings = EvaluationPatchConfig(
    patches_per_scan=3, sampling="center_biased", seed=42,
    left_right_pairs=False,  # D1/rats: optional paired LR views; MouseFull: keep False.
)

model_settings = ModelConfig(
    architecture="3drnet", filters=(16, 32, 64, 128, 256),
    multiscale=True, deep_supervision=True,
    residual=True, attention=True, max_pooling=True, classifier=False,
)
loss_settings = LossConfig(
    regions="dice_ce", gradient_fix_version=1, gamma=0.8, brain_alpha=0.6, batch_wise=True,
    # Total loss = weighted sum of the active output-head losses.
    regions_weight=1.0, brain_weight=1.0, classifier_weight=1.0,
)
# Full volumes: postprocess first, then restore original annotation IDs.
postprocessing_settings = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=True, use_centroids=True,
    default_hemisphere="right", lesion_voxels_threshold=10,
)
inference_settings = InferenceConfig(
    enabled=True, stride=40, save_predictions=True,
    original_label_ids=True, postprocessing=postprocessing_settings,
)
initialization = InitializationConfig()

config = ExperimentConfig(
    data=data_settings, preprocessing=preprocessing, training=training,
    evaluation_patches=evaluation_patch_settings,
    fixed_split=split_settings,
    model=model_settings, labels=labels, loss=loss_settings,
    inference=inference_settings, initialization=initialization,
    learning_rate=INITIAL_LEARNING_RATE, split_seed=42,
    cv_validation_fraction=0.2,
)
config.validate()

## Configure callbacks directly

`MODE="cross_validation"` uses only LR reduction. Fixed-split studies add TensorBoard, early stopping and CSV logging. All variants/seeds receive the same list. Edit the callback constructors or the list in the next cell; the training call explicitly receives `callbacks=callbacks`. No callback is added by a hidden experiment factory. `TrainingConfig.default_callbacks=False` prevents duplicate automatic logging/LR callbacks. Checkpoint selection uses the visible `TrainingConfig.monitor` and `monitor_mode` settings.

Relative log paths resolve inside each run/fold, so folds and study variants do not share files. The runner copies unused callback instances for every fit and saves their settings in `callbacks.json`; resume requires the same saved settings. CSV logs append when resuming.

The runner always saves best weights and epoch recovery state, even when this list is empty. It restores the selected best checkpoint before evaluation. For that reason early stopping uses `restore_best_weights=False`: restoring weights inside its epoch-end hook would mix earlier weights with the latest optimizer in the recovery checkpoint. The final best-weight behavior matches the original workflow.


In [ ]:
import tensorflow as tf
from src.experiments.schedules import CosineEpochSchedule

# Select the policy explicitly; test/holdout metrics never drive these callbacks.
callbacks: list[tf.keras.callbacks.Callback] = []
if USE_VALIDATION_FOR_SELECTION:
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", mode="min", factor=0.5, patience=10,
        min_lr=MINIMUM_LEARNING_RATE, verbose=1,
    )
    callbacks.append(reduce_lr)
    if MODE == "fixed_split":
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", mode="min", patience=20, verbose=1,
            restore_best_weights=False,
        )
        callbacks.append(early_stopping)
else:
    schedule = CosineEpochSchedule(
        initial_learning_rate=config.learning_rate,
        minimum_learning_rate=MINIMUM_LEARNING_RATE,
        epochs=config.training.epochs,
    )
    scheduler = tf.keras.callbacks.LearningRateScheduler(schedule=schedule, verbose=1)
    callbacks.append(scheduler)

csv_filename: str = "training.csv"
csv_logger = tf.keras.callbacks.CSVLogger(filename=csv_filename, append=True)
callbacks.append(csv_logger)
if MODE == "fixed_split":
    log_dir: str = "tensorboard"
    tensorboard = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
    callbacks.append(tensorboard)


## Single-setting experiment matrix

Each variant changes one supported setting. `residual=False` removes shortcuts throughout the network, including any multiscale branches; encoder-decoder skip concatenations remain. `attention=False` removes decoder attention. `max_pooling=False` replaces pooling with strided convolution.

`multiscale=False` removes the three pooled-input feature branches and their encoder concatenations. `deep_supervision=False` removes the coarse projections and their sum from both segmentation heads, retaining each original finest-level projection and final output layer. This ablates fused decoder logits; the original model has no separate auxiliary losses. Both switches default to `True`.

Optional `loss_variants` compare the original probability loss with corrected v1 on the same D1 task. The inactive `gradient_fix_version=1` reference setting specifies which corrected objective the optional row selects. Legacy `dice_ce` is rejected if selected channels have unavailable annotations. Compare segmentation metrics across objectives; loss values have different meanings.

Optional augmentation, cropping and repetition rows change training strategy; repetition changes optimizer updates per epoch. All rows reuse the same saved animals, evaluation patches and seeds. A changed matrix requires a new study folder; resume validates the original matrix. The U-Net comparison belongs in notebook 08.

Every variant starts from fresh random weights and a fresh optimizer. Fixed studies use the selected reference fold; CV repeats the same variants across all reference folds. Animal roles, seeds and evaluation patches are shared across variants.


In [ ]:
from IPython.display import display
from src.experiments.ablation import AblationVariant, ablation_plan

variants: list[AblationVariant] = [
    AblationVariant("reference"),
    AblationVariant("no_residual", "residual", False),
    AblationVariant("no_attention", "attention", False),
    AblationVariant("no_multiscale", "multiscale", False),
    AblationVariant("no_deep_supervision", "deep_supervision", False),
    # AblationVariant("strided_downsampling", "max_pooling", False),
    # AblationVariant("no_augmentation", "augment", False),
    # AblationVariant("fixed_center_crop", "random_crop", False),
    # AblationVariant("no_repetition", "data_multiplier", 1),
]
# Loss rows keep every architecture/training setting at the reference value.
loss_variants: list[AblationVariant] = [
    # AblationVariant("gradient_fix_v1", "regions", "dice_ce_gradient_fix"),
]
variants.extend(loss_variants)

plan = ablation_plan(config, variants)
display(plan)

models_per_variant: int = 1
if MODE == "cross_validation":
    models_per_variant = config.training.k
total_models: int = len(variants) * len(TRAINING_SEEDS) * models_per_variant
print("Maximum fitted models:", total_models)

## One shared animal selection

Load a chosen `SPLIT_FILE` or `FOLD_FILE` for manuscript comparisons. The default selection comes from the shared reference plan. Every variant and training seed receives that same selection; validation/test scans are never multiplied. Confirmed duplicate animals require explicit identity overrides in the reference data configuration.

Set `RESUME_STUDY=True` to continue an existing `STUDY_DIR`. Its saved selection is loaded automatically and the full study contract is checked before execution. Keep the matrix, seed order, configuration and input files unchanged. Completed runs are reused through the shared runner. Fresh studies reject existing directories.

The default `REFERENCE_SPLITS` is `DATA_ROOT.parent / "D1_splits_final.json"`.
The reference stores the original `train` and `test` animal lists.
Original D1 memberships are preserved without modality rebalancing;
optional inner validation comes only from training. Superseded files are backed up.
Fixed runs use `REFERENCE_FOLD=4`; K-Fold runs use the complete reference plan.
MouseFull adds balanced SHAM folds while preserving every updated D1 role.
Explicit `SPLIT_FILE`/`FOLD_FILE` and resumed memberships take precedence.
Set `REFERENCE_SPLITS=None` to generate a fresh plan. Mouse experiments start
from scratch on these memberships; weights are not shared across folds.


In [ ]:
from src.experiments.data import prepare_data
from src.experiments.ablation import study_selection

prepared = prepare_data(config)
selection = study_selection(
    data=prepared, mode=MODE, study_dir=STUDY_DIR,
    split_file=SPLIT_FILE, fold_file=FOLD_FILE, resume=RESUME_STUDY,
    reference_file=REFERENCE_SPLITS, reference_fold=REFERENCE_FOLD,
)
split = selection.split
folds = selection.folds
summary = selection.summary
coverage = prepared.annotation_summary()

display(summary)
display(coverage)


## Check a common validation batch

This preview uses the reference configuration and does not build any models. Every variant retains the same input and output contract; only selected classifier ablations change the auxiliary output list.

In [ ]:
validation_ids = []
if split is not None:
    validation_ids = list(split.validation)
else:
    validation_ids = folds[0].val_ids

validation_generator = prepared.make_generator(validation_ids, training=False, batch_size=1)
images, targets = validation_generator[0]
print("Input:", images.shape)
print("ROI targets:", targets[0].shape)
print("Brain targets:", targets[1].shape)
del images, targets, validation_generator

## Run or resume the matched study

The shared runner compiles each configuration, manages checkpoints and restores validation-selected best weights before full-volume test/outer-holdout evaluation. The study writes its matrix, source/configuration contract, shared split and run index. Each variant/seed retains its own complete experiment artifacts and recovery state. No expensive training starts while `RUN_STUDY=False`.

In [ ]:
from src.experiments.ablation import run_ablation

study = None
if RUN_STUDY:
    study = run_ablation(
        prepared, variants, study_dir=STUDY_DIR, mode=MODE,
        split=split, folds=folds, seeds=TRAINING_SEEDS, resume=RESUME_STUDY, callbacks=callbacks,
    )
    print("Study saved to:", study.study_dir)
else:
    print("Matrix and data checked. Set RUN_STUDY = True to run or resume.")

## Inspect descriptive results

Compare full-volume measurements, statuses, actual training/inference time and complexity before interpreting changes. The plot below uses one scan-average per animal, variant, seed and fold; seed/fold identities remain visible. Repeated models are not extra independent animals. This notebook performs no significance tests or automatic hyperparameter selection; animal-aware paired analyses and uncertainty belong in the revision-results notebook.

`full_volume_metrics.csv` retains raw scan rows and unavailable values. `runs.json` links each variant/seed to its saved configuration. `timings.csv` retains per-fold costs. Predictions and detailed histories remain inside each run folder.

In [ ]:
import plotly.express as px
import plotly.io as pio
from src.experiments.ablation import animal_metric_values

pio.renderers.default = "vscode"
evaluation_role: str = "test"
if MODE == "cross_validation":
    evaluation_role = "holdout"

if study is not None:
    display(study.timings)
    dice = animal_metric_values(study, metric="Dice", role=evaluation_role)
    display(dice)
    if not dice.empty:
        figure = px.box(
            dice, x="Variant", y="Value", color="Variant", facet_col="Region",
            points="all", hover_data=["Subject", "Seed", "Fold", "Scans"],
            title="Descriptive full-volume Dice by animal and model",
        )
        figure.show()

## Best-checkpoint patch and full-volume measurements

Native patch losses and metrics use every saved deterministic validation/test patch. These are cohort aggregates, not independent animal scores. Full-volume brain and region scores use postprocessed predictions, original annotation IDs and valid references. The workbook contains separate sheets for the two evaluation stages.


In [ ]:
import pandas as pd

if study is not None:
    display(study.patch_metrics)
    full_volume_metrics = study.metrics
    evaluation_folder = study.study_dir
    selected_columns = ["DatasetRole", "Fold", "Head", "Region", "Metric", "Value", "Status"]
    if not full_volume_metrics.empty:
        display(full_volume_metrics[selected_columns].head(30))
    from src.evaluation.classification import classification_report, healthy_lesion_report
    classifier_report_result = classification_report(study.classifier_predictions)
    if not classifier_report_result.summary.empty:
        display(classifier_report_result.summary)
        display(classifier_report_result.confusion)
    sham_animals, sham_summary = healthy_lesion_report(full_volume_metrics)
    if not sham_summary.empty:
        display(sham_summary)
    from src.evaluation.reporting import evaluation_report

    report_manifest = prepared.subjects.manifest(prepared.subjects.subject_ids)
    detailed_report = evaluation_report(full_volume_metrics, manifest=report_manifest)
    display(detailed_report.region_set_summary)
    display(detailed_report.region_coverage)
    display(detailed_report.subgroup_summary)
    print("Additional anatomy, cohort, participant and structure-size tables are in the workbook.")
    print("Complete native patch and postprocessed volume tables:", evaluation_folder)
    print("Workbook:", evaluation_folder / "evaluation_metrics.xlsx")
else:
    print("Use the final best-checkpoint cell to evaluate a stopped or saved run.")


## Explore test predictions against ground truth

Run the cell below after training/evaluation, or point `VIEW_RUN` at an existing matching run folder to inspect it without fitting. Saved full-volume `regions.nii.gz` files are loaded directly; no predictions are recomputed. This requires inference and prediction saving to have been enabled.

The table lists every available scan/timepoint and fold. The viewer initially loads up to three rows; set `VIEW_SCAN_INDICES=(0, 4)` to explore other rows. Use the Plotly dropdown to select a scan and reference-oriented plane, then drag the slice slider. Zoom/pan works in each MRI/GT/prediction panel.

Both overlays use the same class colors. The title lists unannotated GT regions: their absence is not evidence of a prediction error. Set `VIEW_REGION` to a foreground **original annotation ID** (for example 13 for the ipsilateral ventricle) to isolate one region. `VIEW_OPACITY` controls the overlay. Previews sample up to 64 slices per plane and 192 pixels per side; set `VIEW_MAX_SLICES=None` for every slice and raise `VIEW_MAX_PIXELS` for more detail. Saved masks are unchanged. The controls use native Plotly rendering.

`VIEW_MEMBER` selects a variant/seed from the printed study run keys. The first member is the default. Fixed-split studies display test scans; CV studies display outer holdouts.


In [ ]:
from src.evaluation.inference import saved_prediction_manifest, prediction_comparison_figure

VIEW_RUN: Path | None = None  # Existing run folder, or use the result above.
VIEW_SCAN_INDICES: tuple[int, ...] | None = None  # None: first three manifest rows.
VIEW_REGION: int | None = None  # Original annotation ID (e.g. 13 = ipsi ventricle); None shows all.
VIEW_OPACITY: float = 0.55
VIEW_MAX_SLICES: int | None = 64
VIEW_MAX_PIXELS: int = 192
VIEW_MEMBER: str | None = None  # For example, a key printed below.

viewer_role = "test"
if MODE == "cross_validation":
    viewer_role = "holdout"

viewer_run = VIEW_RUN
if viewer_run is None and study is not None:
    print("Available study runs:", list(study.runs))
    viewer_member = VIEW_MEMBER
    if viewer_member is None:
        viewer_member = next(iter(study.runs))
    viewer_run = study.runs[viewer_member].run_dir

if viewer_run is None:
    print("Train/evaluate first, or set VIEW_RUN to an existing run folder.")
else:
    prediction_views = saved_prediction_manifest(prepared, viewer_run, role=viewer_role)
    display(prediction_views)

    if prediction_views.empty:
        print("No saved predictions for this role. Check the run, study mode and prediction-saving settings.")
    else:
        prediction_figure = prediction_comparison_figure(
            data=prepared, predictions=prediction_views,
            scan_indices=VIEW_SCAN_INDICES, region=VIEW_REGION,
            opacity=VIEW_OPACITY, max_slices=VIEW_MAX_SLICES,
            max_pixels=VIEW_MAX_PIXELS,
        )
        prediction_figure.show(renderer="vscode")


## Evaluate the best checkpoint after stopping training

Interrupt training, then run this cell independently of `result`. Set `CHECKPOINT_RUN` to the existing **run folder**, select `CHECKPOINT_FOLD`, and enable `RUN_CHECKPOINT_EVALUATION`. After a kernel restart, only the repository setup cell and this cell are needed: the saved experiment supplies architecture, preprocessing, label policy and animal selection.

The default cohort matches this notebook. Fixed-split runs use test animals, CV uses outer holdouts, and historical rat CV uses validation animals. Every selected animal includes all timepoints. A best checkpoint must already exist; an unfinished epoch is not evaluated. The committed selected-weight copy is used when recovery state is available.

This measures native patches and performs full-volume inference, saves metrics/predictions to a fresh folder outside the training run, and opens the MRI/GT/prediction viewer. Training weights, optimizer, history and completion flags are left intact. Inference can take time; leave the switch False to skip it. Use notebook 09 to reopen an evaluation without recomputing predictions.

The common RARE reference is explicitly used for checkpoint evaluation, including older checkpoints. A model trained with a different orientation receives a changed input convention; those scores should be interpreted as evaluation with an orientation override. Training recovery files remain unchanged.


The independent cell exposes `checkpoint_postprocessing` and always restores original annotation IDs, including when the saved run predates these settings. It writes `prediction_labels.json`, so the viewer uses the exact exported IDs. `CHECKPOINT_REGION` selects an original annotation ID. Changing evaluation settings does not change training targets or weights.


In [ ]:
from src.experiments.config import PostprocessingConfig
from src.evaluation.inference import (
    best_checkpoint_settings, evaluate_saved_model,
    saved_prediction_manifest, prediction_comparison_figure,
)

RUN_CHECKPOINT_EVALUATION: bool = True
CHECKPOINT_RUN: Path | None = None  # results/<model>/<run>; required after interruption/restart.
CHECKPOINT_FOLD: int = 0
CHECKPOINT_OUTPUT: Path | None = None  # None creates a fresh evaluation folder.
CHECKPOINT_SCAN_INDICES: tuple[int, ...] | None = None
CHECKPOINT_REGION: int | None = None
CHECKPOINT_MAX_SLICES: int | None = 64
CHECKPOINT_ROLE: str = "test"  # Use holdout for a CV run after a fresh kernel start.
CHECKPOINT_MEMBER: str | None = None  # Available variant/seed key, if study exists above.
if globals().get("MODE") == "cross_validation":
    CHECKPOINT_ROLE = "holdout"

checkpoint_run = CHECKPOINT_RUN
current_study = globals().get("study")
if checkpoint_run is None and current_study is not None:
    print("Available study runs:", list(current_study.runs))
    checkpoint_member = CHECKPOINT_MEMBER
    if checkpoint_member is None:
        checkpoint_member = next(iter(current_study.runs))
    checkpoint_run = current_study.runs[checkpoint_member].run_dir

checkpoint_postprocessing = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=True, use_centroids=True,
    default_hemisphere="right", lesion_voxels_threshold=10,
)

checkpoint_evaluation = None
if RUN_CHECKPOINT_EVALUATION:
    if checkpoint_run is None:
        raise ValueError("Set CHECKPOINT_RUN to the stopped run folder first")
    checkpoint_settings = best_checkpoint_settings(
        run_dir=checkpoint_run, fold=CHECKPOINT_FOLD, role=CHECKPOINT_ROLE,
        output_dir=CHECKPOINT_OUTPUT, reference=REFERENCE,
        original_label_ids=True, postprocessing=checkpoint_postprocessing,
        evaluate_patches=True,
    )
    print("Selected weights:", checkpoint_settings.model_file)
    checkpoint_evaluation = evaluate_saved_model(checkpoint_settings)
    print("Evaluation saved to:", checkpoint_evaluation.output_dir)
    display(checkpoint_evaluation.patch_metrics)
    print("Evaluation workbook:", checkpoint_evaluation.output_dir / "evaluation_metrics.xlsx")
    from src.evaluation.classification import classification_report, healthy_lesion_report
    checkpoint_classifier = classification_report(checkpoint_evaluation.classifier_predictions)
    if not checkpoint_classifier.summary.empty:
        display(checkpoint_classifier.summary)
        display(checkpoint_classifier.confusion)
    checkpoint_sham_animals, checkpoint_sham_summary = healthy_lesion_report(checkpoint_evaluation.metrics)
    if not checkpoint_sham_summary.empty:
        display(checkpoint_sham_summary)
    checkpoint_metrics = checkpoint_evaluation.metrics
    checkpoint_dice = checkpoint_metrics.loc[checkpoint_metrics["Metric"] == "Dice"]
    display(checkpoint_dice)
    from src.evaluation.reporting import evaluation_report

    checkpoint_report = evaluation_report(
        checkpoint_metrics, manifest=checkpoint_evaluation.manifest,
    )
    display(checkpoint_report.region_set_summary)
    display(checkpoint_report.region_coverage)
    display(checkpoint_report.subgroup_summary)
    display(checkpoint_report.participant_flow)
    checkpoint_views = saved_prediction_manifest(
        data=checkpoint_evaluation.data, run_dir=checkpoint_evaluation.output_dir,
        role=CHECKPOINT_ROLE,
    )
    display(checkpoint_views)
    checkpoint_figure = prediction_comparison_figure(
        data=checkpoint_evaluation.data, predictions=checkpoint_views,
        scan_indices=CHECKPOINT_SCAN_INDICES, region=CHECKPOINT_REGION,
        max_slices=CHECKPOINT_MAX_SLICES,
    )
    checkpoint_figure.show(renderer="vscode")
else:
    print("Set CHECKPOINT_RUN and RUN_CHECKPOINT_EVALUATION=True to evaluate saved best weights.")


## Paper tables: average test performance across folds

The main dataframe reports **mean of fold animal-means ± SD across folds**. Longitudinal scans are averaged within each animal first. A single-fold evaluation shows a mean without an invented zero fold SD. Seeds, datasets, SHAM/TBI and evaluation roles stay separate.

Set `PAPER_RESULTS_DIR` to an explicit completed run, study or checkpoint-evaluation folder to load it after a kernel restart. Otherwise the cell uses the current study/run, then a checkpoint evaluation. It reads saved CSVs and performs no fitting or inference. Review `fold_coverage` before copying a final table.

`paper_table` contains full-volume brain/region results. `paper_region_table` contains explicit region-set means; by default these require complete annotation for the selected set. Individual ROI rows still use all valid partial annotations. `Animals` counts Dice contributors, `BoundaryAnimals` counts defined boundary measurements, and the numerical `statistics` table contains every metric's denominator. Missing values stay missing.

`paper_animal_intervals` gives the separate animal mean with bootstrap confidence intervals. It is not an interval for the equal-fold estimator. Native patch measurements, classification/confusion, volume agreement, reference comparisons and recorded costs remain separate tables. Paired comparisons use the chosen reference and Holm adjustment over the returned test family. Classifier exact intervals pool disjoint holdouts only; repeated test/external predictions do not add animals.

Enable `EXPORT_PAPER_TABLES` to write CSVs and `paper_tables.xlsx` to a fresh folder. Full-precision numerical results, source hashes and aggregation settings are included. Patch losses retain their own objective definitions. This does not complete the deferred independent D4 manual-volume analysis.


In [ ]:
from pathlib import Path
from IPython.display import display
from src.evaluation.paper_tables import PaperTableConfig, paper_tables_from_folder

PAPER_RESULTS_DIR: Path | None = Path("results/ablations/d1_3drnet_components")  # Explicit run, study or checkpoint-evaluation folder.
PAPER_ROLES: tuple[str, ...] = ("holdout", "test")
PAPER_REFERENCE_VARIANT: str | None = "reference"  # None skips paired comparisons.
PAPER_BOOTSTRAP_SAMPLES: int = 2000
COMPLETE_REGION_SETS_ONLY: bool = True
EXPORT_PAPER_TABLES: bool = False
PAPER_OUTPUT_DIR: Path | None = None  # None: <selected folder>/paper_tables; must be fresh.

paper_folder = PAPER_RESULTS_DIR
if paper_folder is None:
    current_study = globals().get("study")
    current_result = globals().get("result")
    current_checkpoint = globals().get("checkpoint_evaluation")
    current_evaluation = globals().get("evaluation")
    if current_study is not None:
        paper_folder = current_study.study_dir
    elif current_result is not None:
        paper_folder = current_result.run_dir
    elif current_checkpoint is not None:
        paper_folder = current_checkpoint.output_dir
    elif current_evaluation is not None and hasattr(current_evaluation, "output_dir"):
        paper_folder = current_evaluation.output_dir

paper_tables = None
paper_table = None
paper_region_table = None
paper_animal_intervals = None
if paper_folder is None:
    print("Finish evaluation or set PAPER_RESULTS_DIR to saved results; no training is required.")
else:
    paper_settings = PaperTableConfig(
        roles=PAPER_ROLES, reference_variant=PAPER_REFERENCE_VARIANT,
        complete_region_sets_only=COMPLETE_REGION_SETS_ONLY,
        bootstrap_samples=PAPER_BOOTSTRAP_SAMPLES, confidence=0.95, seed=42, decimals=3,
    )
    print("Paper-table source:", paper_folder)
    paper_tables = paper_tables_from_folder(paper_folder, settings=paper_settings)
    paper_table = paper_tables.segmentation
    paper_region_table = paper_tables.region_sets
    paper_animal_intervals = paper_tables.animal_intervals
    display(paper_tables.fold_coverage)
    display(paper_table)
    display(paper_region_table)
    display(paper_animal_intervals)
    display(paper_tables.patches)
    display(paper_tables.classification)
    display(paper_tables.confusion)
    display(paper_tables.agreement)
    display(paper_tables.comparisons)
    display(paper_tables.healthy_lesion)
    display(paper_tables.costs)

if EXPORT_PAPER_TABLES:
    if paper_tables is None:
        raise ValueError("Build the paper tables before exporting")
    paper_output = PAPER_OUTPUT_DIR
    if paper_output is None:
        paper_output = Path(paper_folder) / "paper_tables"
    saved_paper_tables = paper_tables.export(paper_output)
    print("Paper tables saved to:", saved_paper_tables)
